# Pedestrian Behaviour Prediction Based on Prospect Theory
**MSc Research Project — University of Exeter**
**Author: Taylor Xu | Supervisor: Dr Xingchen Zhang**

---

## Overview

Extends ST CrossingPose (Zhang et al., IEEE TITS 2022) with a two-stage
differentiable Prospect Theory module:

- **Training:** learns a general risk profile θ* across all training pedestrians
- **Inference:** adapts θ* to an individual profile θ_i from a 2-second input clip

Dataset: **JAAD**

## Project Structure

```
files/
├── data/
│   ├── preprocess_jaad.py    Parse JAAD XML annotations → processed splits
│   └── extract_skeletons.py  Re-extract skeletons with ViTPose or OpenPose
├── JAAD/                     Cloned from github.com/ykotseruba/JAAD
├── jaad_dataset.py           JAAD PyTorch Dataset class
├── stgcn_backbone.py         ST-GCN encoder (adapted from ST CrossingPose)
├── prospect_theory.py        Two-stage PT: general profile + adaptation network
├── pt_behaviour.py  Full model: backbone + PT gate + classifier
├── train.py                  Training loop
├── evaluate.py               Four-condition ablation + consistency metric
├── extract_skeletons.py      (root copy — use data/ version instead)
├── requirements.txt
└── README.md
```

Processed data layout written by `preprocess_jaad.py`:

```
data/processed_jaad/
  {train,val,test}/
    labels.csv                track_id, label, wait_time, scene_type
    bboxes/<tid>.npy          (T, 4)  [x, y, w, h]
    contexts/<tid>.npy        (T, 5)  [occlusion, traffic_light, vehicle_act,
                                       intersection, signalized]
    frame_ids/<tid>.npy       (T,)    JAAD frame indices (used by extract_skeletons.py)
    skeletons_vitpose/<tid>.npy   (T, 17, 2)  — after running extract_skeletons.py
    skeletons_openpose/<tid>.npy  (T, 17, 2)  — after running with --method openpose
```

## Data Availability

Raw JAAD video clips and annotations are **not** bundled with this submission (the clips alone are ~3.1GB, well beyond what's practical to submit as coursework). They are publicly available from the dataset authors and can be fetched directly:

- **Video clips (MP4, ~3.1GB):** [JAAD_clips.zip](http://data.nvision2.eecs.yorku.ca/JAAD_dataset/data/JAAD_clips.zip)
- **Annotations + Python data interface (`jaad_data.py`):** [github.com/ykotseruba/JAAD](https://github.com/ykotseruba/JAAD)
- **Dataset project page (stats, license, citation):** [data.nvision2.eecs.yorku.ca/JAAD_dataset](https://data.nvision2.eecs.yorku.ca/JAAD_dataset/)
- **Reference benchmark code showing the JAADbeh / JAADall split logic:** [github.com/ykotseruba/PedestrianActionBenchmark](https://github.com/ykotseruba/PedestrianActionBenchmark)

JAADbeh and JAADall are not separate downloads — both are derived from the same JAAD release via `jaad_data.py`, filtered differently: **JAADbeh** keeps only the 618 pedestrians with behavioural crossing annotations (78% crossing rate), while **JAADall** keeps all 2,404 pedestrians visible in the scene, including unannotated bystanders (78% not-crossing rate). `data/preprocess_jaad.py` in this submission produces both subsets from the same cloned `JAAD/` directory; see `--split_type` in that script.

Licensing: JAAD videos and annotations are released under the MIT License by the dataset authors (Rasouli, Kotseruba & Tsotsos, York University). Redistributing a re-packaged copy alongside this submission was avoided both for size and to respect the authors' own distribution channel.

## Setup

Run the cells below in order. Each cell is a runnable shell command (uses the `%%bash` cell magic).

In [ ]:
%%bash
# 1. Clone JAAD data interface (already done)
git clone https://github.com/ykotseruba/JAAD

In [ ]:
%%bash
# 2. Install dependencies
# Note: mmcv-full was renamed to mmcv in v2.x; mmengine and mmdet are also required.
pip3 install -r requirements.txt

In [ ]:
%%bash
# 3. Preprocess JAAD into expected directory structure
python3 data/preprocess_jaad.py \
    --jaad_dir JAAD \
    --output_dir data/processed_jaad

In [ ]:
%%bash
# 4. Download JAAD video clips (required for skeleton extraction)
#    Run the script inside the cloned JAAD repo:
bash JAAD/download_clips.sh

In [ ]:
%%bash
# 5. Re-extract skeletons with ViTPose (recommended)
for split in train val test; do
  python3 data/extract_skeletons.py \
      --video_dir  JAAD/JAAD_clips \
      --tracks_dir data/processed_jaad/$split \
      --output_dir data/processed_jaad/$split/skeletons_vitpose \
      --method vitpose
done

In [ ]:
%%bash
# Repeat with --method openpose for ablation comparison
for split in train val test; do
  python3 data/extract_skeletons.py \
      --video_dir  JAAD/JAAD_clips \
      --tracks_dir data/processed_jaad/$split \
      --output_dir data/processed_jaad/$split/skeletons_openpose \
      --method openpose
done

In [ ]:
%%bash
# 6. Compare skeleton quality (optional)
python3 data/extract_skeletons.py --compare_only \
    --vitpose_dir  data/processed_jaad/train/skeletons_vitpose \
    --openpose_dir data/processed_jaad/train/skeletons_openpose

## Training

In [ ]:
%%bash
# Phase 1 — JAAD, ViTPose skeletons (recommended)
python3 train.py \
    --data_dir data/processed_jaad \
    --dataset jaad --skeleton_src vitpose \
    --output_dir experiments/run_vitpose \
    --epochs 100 --batch_size 32 --lr 1e-3 --device cuda

In [ ]:
%%bash
# Ablation — OpenPose skeletons (for skeleton quality comparison)
python3 train.py \
    --data_dir data/processed_jaad \
    --dataset jaad --skeleton_src openpose \
    --output_dir experiments/run_openpose

## Evaluation

In [ ]:
%%bash
python3 evaluate.py \
    --data_dir data/processed_jaad \
    --checkpoint experiments/run_vitpose/best_model.pt \
    --skeleton_src vitpose --device cuda

Automatically reports three ablation conditions:
1. **Full model** — ST-GCN + two-stage PT (general + individual adaptation)
2. **General only** — Δθ = 0, no individual adaptation
3. **No PT** — gating disabled

Run again with `--skeleton_src openpose` for the fourth condition.

## Two-Stage PT Mechanism

```
TRAINING
  All pedestrians in batch
      → PT feature vector (wait_time, TTC, scene_type, head_orient, speed)
      → Shared general profile θ* = (α*, β*, λ*, γ*)
      → Learned as globally shared nn.Parameter values

INFERENCE  (2-second clip)
  Input clip
      → Adaptation network
      → Δθ_i (individual deviation, constrained to ±20% of parameter range)
      → θ_i = θ* + Δθ_i
      → CPT utility U_i = w(p_success)*V(gain) + w(p_collision)*V(-loss)
      → U_i gates ST-GCN embedding via CrossAttentionGate
      → Classifier → P(crossing)
```

## Key References

- Zhang X., Angeloudis P. & Demiris Y. (2022). ST CrossingPose. *IEEE TITS*.
- Tversky A. & Kahneman D. (1992). Advances in Prospect Theory. *J. Risk & Uncertainty*.
- Xu Y. et al. (2022). ViTPose. *NeurIPS*.
- Kotseruba I. et al. (2021). Benchmark for pedestrian action prediction. *WACV*.
- Rasouli A. et al. (2017). JAAD. *ICCVW*.
- Rasouli A. et al. (2019). PIE. *ICCV*.